# ML01：手写线性/逻辑回归与正则化

从 NumPy loss/gradient 到 validation 模型选择，避免只会调用 fit。

**运行方式**：上传本 `.ipynb` 到 Google Colab，连接 Python runtime，从上到下运行。
默认 CPU 即可；本 notebook 自带代码和合成数据，不需要克隆仓库、上传论文、挂载 Drive 或配置本地环境。
所有输出留空，首次运行后才产生实际结果；不要把材料准备状态当作已复现实验。

如果所选云端 runtime 缺少包，可在 Colab 新单元执行 `%pip install numpy matplotlib pandas scikit-learn torch`，
然后重启 runtime。不要为了本课程在本机安装依赖。


## 1. 环境与随机性
记录版本和 seed。换 seed 时从本单元重新运行；图中用英文标签避免依赖额外中文字体。


In [ ]:
import json
import math
import platform
import random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
SEED = 7
random.seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)
print('Python:', platform.python_version(), '| NumPy:', np.__version__, '| seed:', SEED)
import sklearn
print('sklearn:', sklearn.__version__)


## 2. 手写 linear MSE gradient 与有限差分
使用半 MSE，使梯度为 X.T@(Xw-y)/N。y 与 prediction 都是 [N,1]。


In [ ]:
X = rng.normal(size=(400, 3)) * np.array([1., 8., 0.05])
y = X @ np.array([[1.], [0.3], [-2.]]) + rng.normal(scale=0.2, size=(400, 1))
train_ids, val_ids = np.arange(280), np.arange(280, 400)
Xtr, Xva, ytr, yva = X[train_ids], X[val_ids], y[train_ids], y[val_ids]
def linear_loss_grad(x, target, weights):
    prediction = x @ weights
    assert prediction.shape == target.shape
    error = prediction - target
    return float(0.5 * np.mean(error**2)), x.T @ error / len(x)
w = rng.normal(size=(3, 1))
objective, analytic = linear_loss_grad(Xtr, ytr, w)
numerical = np.zeros_like(w)
eps = 1e-5
for j in range(len(w)):
    delta = np.zeros_like(w); delta[j, 0] = eps
    numerical[j, 0] = (linear_loss_grad(Xtr, ytr, w+delta)[0] -
                       linear_loss_grad(Xtr, ytr, w-delta)[0]) / (2*eps)
assert np.allclose(analytic, numerical, rtol=1e-4, atol=1e-5)
print('linear gradient max error:', np.max(np.abs(analytic-numerical)))


## 3. 同样 LR/steps，仅改变 feature scale
Scaler 只 fit train；intercept 用额外一列 1 表示。图是优化轨迹，不是统计泛化胜负。


In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler().fit(Xtr)
histories = {}
for name, x_train, x_valid in [('raw', Xtr, Xva),
                              ('standardized', scaler.transform(Xtr), scaler.transform(Xva))]:
    a = np.column_stack([x_train, np.ones(len(x_train))])
    b = np.column_stack([x_valid, np.ones(len(x_valid))])
    weights = np.zeros((a.shape[1], 1))
    trace = []
    for step in range(1000):
        value, gradient = linear_loss_grad(a, ytr, weights)
        weights -= 0.01 * gradient
        if step % 10 == 0:
            trace.append((step, value, linear_loss_grad(b, yva, weights)[0]))
    histories[name] = np.asarray(trace)
for name, trace in histories.items():
    plt.plot(trace[:, 0], trace[:, 2], label=name)
plt.yscale('log'); plt.xlabel('optimizer steps'); plt.ylabel('validation half-MSE')
plt.legend(); plt.show()


## 4. 手写 logistic：stable BCE 与 p-y 梯度


In [ ]:
Xc = rng.normal(size=(700, 3))
Xc = np.column_stack([Xc, np.ones(len(Xc))])
p_true = 1 / (1 + np.exp(-(Xc @ np.array([1.5, -1., .5, -.2]))))
yc = (rng.random(len(Xc)) < p_true).astype(float).reshape(-1, 1)
wc = np.zeros((4, 1)); logistic_trace = []
for step in range(500):
    logits = Xc[:450] @ wc
    probabilities = 1 / (1 + np.exp(-np.clip(logits, -40, 40)))
    value = np.mean(np.logaddexp(0, logits) - yc[:450] * logits)
    gradient = Xc[:450].T @ (probabilities-yc[:450]) / 450
    wc -= 0.1 * gradient
    logistic_trace.append(float(value))
held_logits = Xc[450:] @ wc
held_loss = float(np.mean(np.logaddexp(0, held_logits)-yc[450:]*held_logits))
print('held-out logistic NLL:', held_loss)
plt.plot(logistic_trace); plt.xlabel('step'); plt.ylabel('training BCE'); plt.show()


## 5. Polynomial + Ridge：先 validation 选择，再 test
以下是另一份独立合成回归数据。test 不参与 degree/penalty 选择。


In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
xp = rng.uniform(-1, 1, size=(160, 1))
yp = np.sin(3*xp[:, 0]) + rng.normal(scale=0.2, size=160)
order = rng.permutation(160); tr, va, te = order[:80], order[80:120], order[120:]
candidates, rows = {}, []
for degree in (1, 3, 9, 15):
    for penalty in (0.001, 1., 50.):
        model = make_pipeline(PolynomialFeatures(degree, include_bias=False),
                              StandardScaler(), Ridge(alpha=penalty))
        model.fit(xp[tr], yp[tr])
        candidates[(degree, penalty)] = model
        rows.append({'degree': degree, 'penalty': penalty,
                     'train_mse': mean_squared_error(yp[tr], model.predict(xp[tr])),
                     'val_mse': mean_squared_error(yp[va], model.predict(xp[va]))})
table = pd.DataFrame(rows).sort_values('val_mse'); display(table)
best = table.iloc[0]; key = (int(best['degree']), float(best['penalty']))
selected = candidates[key]
test_mse = float(mean_squared_error(yp[te], selected.predict(xp[te])))
results = {'seed': SEED, 'gradient_max_error': float(np.max(np.abs(analytic-numerical))),
           'logistic_held_out_nll': held_loss, 'selected_degree': key[0],
           'selected_penalty': key[1], 'selected_test_mse': test_mse}
print(results)
grid = np.linspace(-1, 1, 200).reshape(-1, 1)
plt.scatter(xp[tr, 0], yp[tr], s=14, label='train')
plt.plot(grid[:, 0], selected.predict(grid), label='validation-selected model')
plt.xlabel('x'); plt.ylabel('y'); plt.legend(); plt.show()


## 面试追问与实验记录
1. 手推 p-y；为什么 BCE 接 logits 更稳定？
2. 只改变 degree/penalty，解释 bias/variance。
3. 为什么在 validation 上挑最小值后还需要独立 test？

记录 Question / Data / Config / Metric / Observation / Explanation / Limitation。
改变实验设置时不要反复用 test 选择答案；保存 notebook 后再下载 `.ipynb` 保留输出。


## 可选：导出指标
默认关闭。需要时打开开关；文件只写到当前 **Colab runtime** 的 `outputs/`。runtime 临时文件不等于已永久保存，可使用 Colab 文件面板下载。


In [ ]:
EXPORT_RESULTS = False
if EXPORT_RESULTS:
    destination = Path('outputs')
    destination.mkdir(exist_ok=True)
    (destination / 'metrics.json').write_text(
        json.dumps(results, indent=2, ensure_ascii=False, allow_nan=False), encoding='utf-8')
    print('Saved outputs/metrics.json in the Colab runtime; download it from the Files panel.')
